In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
dfMinute = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl', compression=None)
df_sentiment = df_sentiment.loc[df_sentiment["Date"]>"2020-05-31"]
df_sentiment["Date"]= pd.to_datetime(df_sentiment["Date"])

In [ ]:

df_sentiment.rename(columns={"sentiment":"Sentiment",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                "SentimentOrig_ln_return":"SentimentOrig Natural Log",
                "SentimentOrig_ln_return(t-1)":"SentimentOrig Natural Log(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive Natural Log",
                "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative Natural Log", 
                "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral Natural Log",
                "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)"}
                ,inplace=True)

In [ ]:
df_sentiment.columns.values

# Title

In [ ]:
import unicodedata
dividend=0
partner=0
earnings=0
acquisition=0
public_offering=0
notes_offering=0
alliance=0
merge=0
appointment=0
ceo_compensation=0
government=0
new_brand=0
sales_down=0
covid19=0
legal_action=0
sales=0
agreement=0
undefined=0
rows_list = []

for date, row in df_sentiment.T.iteritems():
    sentence = unicodedata.normalize('NFKD', row['Title'])
    # print(date)
    if ("DIVID" in sentence):
        Type = "DIVIDEND"
        dividend+=1
    elif ("EARNING" in sentence):
        Type = "EARNING"
        earnings+=1
    elif (" EPS " in sentence):
        Type = "EARNING"
        earnings+=1    
    elif ("ACQUI" in sentence):
        Type="ACQUI"
        acquisition+=1
    elif ("PUBLIC OFFERING" in sentence):
        Type = "PUBLIC OFFERING"
        public_offering+=1
    elif ("AGREEMENT" in sentence):
        Type="AGREEMENT"
        agreement+=1
    elif ("NOTES OFFERING" in sentence):
        Type = "NOTES OFFERING"
        notes_offering+=1
    elif ("ALLIANCE" in sentence):
        Type = "ALLIANCE"
        alliance+=1
    elif ("TEAM UP" in sentence):
        Type="TEAM UP"
        alliance+=1
    elif ("MERGE" in sentence):
        Type = "MERGE"
        merge+=1
    elif ("PARTNER" in sentence):
        Type = "PARTNER"
        partner+=1
    elif ("APPOINT" in sentence):
        Type = "APPOINTMENT"
        appointment+=1
    elif ("TOTAL COMPENSATION" in sentence):
        Type = "CEO COMPENSATION"
        ceo_compensation+=1
    elif ("GOVERNMENT" in sentence):
        Type="GOVERNMENT"
        government+=1
    # elif ("SENIOR NOTES" in sentence):
    #     sales_notes+=1
    elif ("SALES" in sentence):
        Type="SALES"
        sales+=1
    # elif("COVID-19" in sentence):
    #     Type="COVID-19"
    #     covid19+=1
    #     print(sentence)
    #     print("--------------")
    # # elif("LEGAL ACTION" in sentence):
    # #     legal_action+=1
    
    else:
        Type="UNDEFINED"
        undefined+=1

    print(sentence)
        # print("--------------")
    #     ['Time', 'Date', 'Hour', 'Negative', 'Positive', 'Sentiment', 'Polarity',
    #    'Subjectivity', 'PolarityB', 'Articles', 'Title', 'Text', 'Company'
    new_row = {"Time":row["Time"], "Date":row["Date"], "Hour": row["Hour"], "Negative":row["Negative"], "Positive":row["Positive"], "Sentiment":row["Sentiment"], "Polarity": row["Polarity"], "Subjectivity":row["Subjectivity"], "PolarityB":row["PolarityB"],"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "Type":Type}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Time","Date", "Hour","Negative", "Positive", "Sentiment", "Polarity", "Subjectivity", "PolarityB","Articles","Title","Text", "Company", "Type"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeTitle.pkl")


# TEXT

In [ ]:
import unicodedata
dividend=0
partner=0
earnings=0
acquisition=0
public_offering=0
notes_offering=0
alliance=0
merge=0
appointment=0
ceo_compensation=0
government=0
new_brand=0
sales_down=0
covid19=0
legal_action=0
sales=0
agreement=0
undefined=0
rows_list = []

for date, row in df_sentiment.T.iteritems():
    sentence = unicodedata.normalize('NFKD', row['Articles'])
    # print(date)
    if ("DIVID" in sentence):
        Type = "DIVIDEND"
        dividend+=1        
        print(row['Title'])
        print("--------------")
    elif ("EARNING" in sentence):
        Type = "EARNING"
        earnings+=1
    elif (" EPS " in sentence):
        Type = "EARNING"
        earnings+=1    
    elif ("ACQUI" in sentence):
        Type="ACQUI"
        acquisition+=1
    elif ("PUBLIC OFFERING" in sentence):
        Type = "PUBLIC OFFERING"
        public_offering+=1
    elif ("AGREEMENT" in sentence):
        Type="AGREEMENT"
        agreement+=1
    elif ("NOTES OFFERING" in sentence):
        Type = "NOTES OFFERING"
        notes_offering+=1
    elif ("ALLIANCE" in sentence):
        Type = "ALLIANCE"
        alliance+=1
    elif ("TEAM UP" in sentence):
        Type="TEAM UP"
        alliance+=1
    elif ("MERGE" in sentence):
        Type = "MERGE"
        merge+=1
    elif ("PARTNER" in sentence):
        Type = "PARTNER"
        partner+=1
    elif ("APPOINT" in sentence):
        Type = "APPOINTMENT"
        appointment+=1
    elif ("TOTAL COMPENSATION" in sentence):
        Type = "CEO COMPENSATION"
        ceo_compensation+=1
    elif ("GOVERNMENT" in sentence):
        Type="GOVERNMENT"
        government+=1
    # elif ("SENIOR NOTES" in sentence):
    #     sales_notes+=1
    elif ("SALES" in sentence):
        Type="SALES"
        sales+=1
    # elif("COVID-19" in sentence):
    #     Type="COVID-19"
    #     covid19+=1
    #     print(sentence)
    #     print("--------------")
    # # elif("LEGAL ACTION" in sentence):
    # #     legal_action+=1
    
    else:
        Type="UNDEFINED"
        undefined+=1

        # print(sentence)
        # print("--------------")
    #     ['Time', 'Date', 'Hour', 'Negative', 'Positive', 'Sentiment', 'Polarity',
    #    'Subjectivity', 'PolarityB', 'Articles', 'Title', 'Text', 'Company'
    new_row = {"Time":row["Time"], "Date":row["Date"], "Hour": row["Hour"], "Negative":row["Negative"], "Positive":row["Positive"], "Sentiment":row["Sentiment"], "Polarity": row["Polarity"], "Subjectivity":row["Subjectivity"], "PolarityB":row["PolarityB"],"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "Type":Type}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Time","Date", "Hour","Negative", "Positive", "Sentiment", "Polarity", "Subjectivity", "PolarityB","Articles","Title","Text", "Company", "Type"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeText.pkl")


In [ ]:
pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeText.pkl")
